In [36]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as mpl
import seaborn as sb
import missingno as msn
import warnings
warnings.filterwarnings('ignore')

print('OK')

OK


In [37]:
train = pd.read_parquet('C:/Users/Deepayan/Documents/MAVERICK/projects/churn-prediction/artifacts/kkbox_final.parquet')
print('OK')

OK


In [38]:
train.shape, train.dtypes, train.memory_usage(deep=True).sum() / 1e6

((970960, 19),
 msno                          str
 is_churn                    int64
 city                      float64
 bd                        float64
 gender                        str
 registered_via            float64
 registration_init_time    float64
 transanction_count        float64
 first_transaction_date    float64
 last_transaction_date     float64
 last_plan_days            float64
 last_plan_price           float64
 avg_amount_paid           float64
 auto_renew_flag           float64
 cancel_count              float64
 total_secs                float64
 num_unq                   float64
 num_100                   float64
 days_active               float64
 dtype: object,
 np.float64(192.35397))

In [39]:
train['msno'].duplicated().sum()

np.int64(0)

In [40]:
train['city'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [41]:
train['registered_via'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [42]:
whether_city_registered_via_have_same_missing_rows = train.loc[train['city'].isnull(), 'msno'].equals(train.loc[train['registered_via'].isnull(), 'msno'])
print(whether_city_registered_via_have_same_missing_rows) 

#checking is the same rows have both city and registered_via missing
#if that is true, it confirms this is not a merge bug

True


In [43]:
train['has_details'] = train['city'].notnull()
pd.crosstab(train['has_details'], train['is_churn'], normalize= 'index') 

#relates the is_churn column with 'city' and 'registered_via' when they are null and not null

is_churn,0,1
has_details,,
False,0.946524,0.053476
True,0.905399,0.094601


-> Users who have signed up and completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 5.3%
-> Users who have signed up but not completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 9.5%

Hence, profile incompleteness does not directly lead to user being churned as nearly double the users have actually churned who have complete profiles than those who have incomplete profiles. 

In [44]:
train.groupby('has_details')['registration_init_time'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True,860967.0,2.013265e+07,30111.744263,20040326.0,20120214.0,20140602.0,20160118.0,20170424.0


This confirms that all users who have 'city' and 'registered_via' details missing also have 'registration_init_time' blank, strongly leading to the conclusion that these users ('msno') do not exist in the members.csv file but since they exist in the merged parquet, they must have records in transactions.csv and user_logs.csv

In [45]:
train.groupby('has_details')['first_transaction_date'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,108210.0,2.017027e+07,947.033560,20150102.0,20170308.0,20170316.0,20170324.0,20170331.0
True,825368.0,2.017011e+07,1838.337256,20150101.0,20170306.0,20170315.0,20170323.0,20170331.0


COUNT

Out of 109993 users with no details, 108210 users have a transaction date in the window which means there are 1783 users with no details who also have zero transactions in this window.

->The mean, median, 25, 75 percentiles all have dates having a gap of nearly 1-2 days for whether the person has details or not. This effectively rules out the assumption that there must have been a legacy account which logged in users during earlier times without details who have somehow managed to not churn till now before introducing a modified subscription plan for new users requiring their details who churn at a faster rate.

MEDIAN (50%)

The min column shows that earliest transactions date back to 2015-01-02 but the median column shows 2017-03-16, much closer to the max column showing 2017-03-31. This effectively means that there have been significantly more transactions in the last 2-3 weeks of a 2 year window.